# Chuẩn hóa — kiểm chứng 1NF / 2NF / 3NF

Notebook này **chứng minh** từng khẳng định trong `normalized_schema.md`.
Mỗi phụ thuộc hàm (FD) được viết trong tài liệu đều có một cell chạy lại được ở đây.

Bổ sung cho `data_model.ipynb` (kiểm chứng star schema) — cùng dữ liệu, hướng mô hình hóa ngược lại.

**Quy ước:** `A → B` (A xác định hàm B) đúng khi mọi giá trị của `A` chỉ ứng với **đúng một** giá trị `B`.

In [1]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

DATA = 'data'
R = lambda f, **k: pd.read_csv(f'{DATA}/{f}', low_memory=False, **k)

cust   = R('customers.csv',        parse_dates=['signup_date'])
geo    = R('geography.csv')
prod   = R('products.csv')
promo  = R('promotions.csv',       parse_dates=['start_date', 'end_date'])
orders = R('orders.csv',           parse_dates=['order_date'])
items  = R('order_items.csv')
pay    = R('payments.csv')
ship   = R('shipments.csv',        parse_dates=['ship_date', 'delivery_date'])
ret    = R('returns.csv',          parse_dates=['return_date'])
rev    = R('reviews.csv',          parse_dates=['review_date'])
inv    = R('inventory.csv',        parse_dates=['snapshot_date'])
wt     = R('web_traffic.csv',      parse_dates=['date'])
sales  = R('sales.csv',            parse_dates=['Date'])
sub    = R('sample_submission.csv', parse_dates=['Date'])


def fd(df, det, dep):
    # Đếm số giá trị của `det` ứng với >1 giá trị `dep` → vi phạm FD det → dep.
    det = [det] if isinstance(det, str) else list(det)
    g = df.dropna(subset=det).groupby(det)[dep].nunique()
    return int((g > 1).sum()), len(g)


def show_fd(df, det, dep, note=''):
    v, t = fd(df, det, dep)
    mark = 'FD ĐÚNG ' if v == 0 else 'FD SAI  '
    d = det if isinstance(det, str) else '+'.join(det)
    print(f'  {mark} {d:>14s} -> {dep:<16s} {v:>5} vi phạm / {t:,} giá trị  {note}')
    return v, t


print('Số dòng mỗi bảng nguồn:')
for n, d in [('customers', cust), ('geography', geo), ('products', prod), ('promotions', promo),
             ('orders', orders), ('order_items', items), ('payments', pay), ('shipments', ship),
             ('returns', ret), ('reviews', rev), ('inventory', inv), ('web_traffic', wt),
             ('sales', sales), ('sample_submission', sub)]:
    print(f'  {n:18s} {len(d):>9,} dòng × {len(d.columns)} cột')

Số dòng mỗi bảng nguồn:
  customers            121,930 dòng × 7 cột
  geography             39,948 dòng × 4 cột
  products               2,412 dòng × 8 cột
  promotions                50 dòng × 10 cột
  orders               646,945 dòng × 8 cột
  order_items          714,669 dòng × 7 cột
  payments             646,945 dòng × 4 cột
  shipments            566,067 dòng × 4 cột
  returns               39,939 dòng × 7 cột
  reviews              113,551 dòng × 7 cột
  inventory             60,247 dòng × 17 cột
  web_traffic            3,652 dòng × 7 cột
  sales                  3,833 dòng × 3 cột
  sample_submission        548 dòng × 3 cột


---
## 1 — Phủ thời gian: mọi bảng giao dịch dừng ở 2022-12-31

In [2]:
for n, s in [('orders.order_date', orders.order_date), ('sales.Date', sales.Date),
             ('returns.return_date', ret.return_date), ('reviews.review_date', rev.review_date),
             ('shipments.ship_date', ship.ship_date), ('inventory.snapshot', inv.snapshot_date),
             ('web_traffic.date', wt.date), ('sample_submission', sub.Date)]:
    print(f'  {n:22s} {s.min().date()} → {s.max().date()}   {s.nunique():>6,} ngày')

  orders.order_date      2012-07-04 → 2022-12-31    3,833 ngày
  sales.Date             2012-07-04 → 2022-12-31    3,833 ngày
  returns.return_date    2012-07-11 → 2022-12-31    3,806 ngày
  reviews.review_date    2012-07-10 → 2022-12-31    3,825 ngày
  shipments.ship_date    2012-07-04 → 2022-12-29    3,831 ngày
  inventory.snapshot     2012-07-31 → 2022-12-31      126 ngày
  web_traffic.date       2013-01-01 → 2022-12-31    3,652 ngày
  sample_submission      2023-01-01 → 2024-07-01      548 ngày


---
## 2 — 1NF: nguyên tử, không nhóm lặp, định danh được từng dòng

### 2.1 `order_items` KHÔNG có khóa tự nhiên hợp lệ

In [3]:
dup = items[items.duplicated(['order_id', 'product_id'], keep=False)].sort_values(['order_id', 'product_id'])
print(f'{len(dup)} dòng thuộc {dup.order_id.nunique()} đơn có (order_id, product_id) trùng:\n')
print(dup.head(6).to_string(index=False))
print(f'\n→ cùng đơn, cùng sản phẩm, KHÁC quantity và unit_price ⇒ không phân biệt được dòng.')
print(f'  Số dòng tối đa trong một đơn: {items.groupby("order_id").size().max()}')
print(f'  Sửa: thêm line_number ⇒ PK (order_id, line_number)')

32 dòng thuộc 16 đơn có (order_id, product_id) trùng:

 order_id  product_id  quantity  unit_price  discount_amount promo_id promo_id_2
    14280         976         1     4019.47              0.0      NaN        NaN
    14280         976         2     3937.99              0.0      NaN        NaN
   113379         786         6      694.34              0.0      NaN        NaN
   113379         786         1      699.37              0.0      NaN        NaN
   215525        1859         5     1896.11              0.0      NaN        NaN
   215525        1859         5     1897.20              0.0      NaN        NaN

→ cùng đơn, cùng sản phẩm, KHÁC quantity và unit_price ⇒ không phân biệt được dòng.
  Số dòng tối đa trong một đơn: 5
  Sửa: thêm line_number ⇒ PK (order_id, line_number)


### 2.2 Nhóm lặp `promo_id` / `promo_id_2`

In [4]:
n1, n2 = items.promo_id.notna().sum(), items.promo_id_2.notna().sum()
print(f'  promo_id   có giá trị : {n1:,} ({n1/len(items)*100:.1f}%)')
print(f'  promo_id_2 có giá trị : {n2:,} ({n2/len(items)*100:.2f}%)')
print(f'  dòng có promo_id == promo_id_2 : {(items.promo_id == items.promo_id_2).sum()}  (⇒ PK junction không đụng độ)')
print(f'  dòng có promo_id_2 nhưng promo_id rỗng : {(items.promo_id_2.notna() & items.promo_id.isna()).sum()}')
print(f'\n→ kích thước order_item_promotion = {n1:,} + {n2:,} = {n1+n2:,} dòng')

pair = items[items.promo_id_2.notna()].groupby(['promo_id', 'promo_id_2']).size()
print('\nCác cặp promo đồng thời:')
print(pair.to_string())

  promo_id   có giá trị : 276,316 (38.7%)
  promo_id_2 có giá trị : 206 (0.03%)
  dòng có promo_id == promo_id_2 : 0  (⇒ PK junction không đụng độ)
  dòng có promo_id_2 nhưng promo_id rỗng : 0

→ kích thước order_item_promotion = 276,316 + 206 = 276,522 dòng

Các cặp promo đồng thời:
promo_id    promo_id_2
PROMO-0013  PROMO-0015    132
PROMO-0023  PROMO-0025     74


### 2.3 Cột dẫn xuất trong `inventory`

In [5]:
d = inv.snapshot_date
print(f'  inventory.year/month suy từ snapshot_date : {bool((d.dt.year.eq(inv.year) & d.dt.month.eq(inv.month)).all())}')
print(f'  inventory.reorder_flag — số giá trị       : {inv.reorder_flag.nunique()} {list(inv.reorder_flag.unique())} → cột chết')

  inventory.year/month suy từ snapshot_date : True
  inventory.reorder_flag — số giá trị       : 1 [np.int64(0)] → cột chết


---
## 3 — 2NF: phụ thuộc bộ phận vào khóa tổ hợp

2NF **chỉ có thể bị vi phạm khi khóa chính là tổ hợp**. Trong 14 file nguồn, chỉ `inventory`
có khóa tổ hợp thực sự — và nó vi phạm ở **cả hai vế** của khóa.

In [6]:
print(f'inventory: PK (snapshot_date, product_id) unique = {not inv.duplicated(["snapshot_date","product_id"]).any()}\n')
print('Phụ thuộc bộ phận vào product_id:')
for c in ['product_name', 'category', 'segment']:
    show_fd(inv, 'product_id', c)
print('\nPhụ thuộc bộ phận vào snapshot_date:')
for c in ['year', 'month']:
    show_fd(inv, 'snapshot_date', c)
print('\n→ 5 cột phải tách khỏi inventory (3 cột → bảng product, 2 cột → dẫn xuất từ ngày)')

inventory: PK (snapshot_date, product_id) unique = True

Phụ thuộc bộ phận vào product_id:
  FD ĐÚNG      product_id -> product_name         0 vi phạm / 1,624 giá trị  
  FD ĐÚNG      product_id -> category             0 vi phạm / 1,624 giá trị  
  FD ĐÚNG      product_id -> segment              0 vi phạm / 1,624 giá trị  

Phụ thuộc bộ phận vào snapshot_date:
  FD ĐÚNG   snapshot_date -> year                 0 vi phạm / 126 giá trị  
  FD ĐÚNG   snapshot_date -> month                0 vi phạm / 126 giá trị  

→ 5 cột phải tách khỏi inventory (3 cột → bảng product, 2 cột → dẫn xuất từ ngày)


### 3.1 Phản chứng: `order_items` KHÔNG vi phạm 2NF

Hai cột dễ bị nghi là phụ thuộc bộ phận — kiểm tra cho thấy không phải.

In [7]:
print('a) unit_price có phụ thuộc riêng product_id không?')
show_fd(items, 'product_id', 'unit_price')
pu = items.groupby('product_id').unit_price.nunique()
print(f'     trung vị {pu.median():.0f} giá khác nhau/sản phẩm, tối đa {pu.max():,}')
print('     → giá là giá TẠI THỜI ĐIỂM BÁN, cần CẢ khóa. Không phải phụ thuộc bộ phận.\n')

print('b) promo_id có phụ thuộc riêng order_id không?')
show_fd(items, 'order_id', 'promo_id', '(chỉ tính dòng khác NULL)')
items['_has'] = items.promo_id.notna()
mix = items.groupby('order_id')._has.nunique()
mo = mix[mix > 1].index
print(f'     NHƯNG {len(mo)} đơn có CẢ dòng mang promo lẫn dòng không mang ⇒ FD thất bại khi tính NULL.')

s = (items[items.order_id.isin(mo)]
     .merge(prod[['product_id', 'category']], on='product_id')
     .merge(promo[['promo_id', 'applicable_category']], on='promo_id', how='left'))
w = s[s.promo_id.notna()]
print(f'     Trong {len(mo)} đơn đó, dòng MANG promo:')
print(f'       promo có applicable_category      : {w.applicable_category.notna().mean()*100:.0f}%')
print(f'       category sản phẩm == applicable   : {(w.category == w.applicable_category).mean()*100:.0f}%')
print('     → promo giới hạn category chỉ dính vào dòng đủ điều kiện ⇒ promo là thuộc tính CẤP DÒNG.')
print('       Junction table phải ở grain dòng hàng, không phải grain đơn.')

print('\nc) discount_amount có phụ thuộc riêng order_id hay product_id không?')
show_fd(items, 'order_id', 'discount_amount')
show_fd(items, 'product_id', 'discount_amount')
print('     → cần CẢ khóa (chiết khấu phụ thuộc promo × giá trị dòng). Không phải phụ thuộc bộ phận.')
print('\n→ order_item KHÔNG vi phạm 2NF: cả 3 cột không khóa đều cần đủ (order_id, line_number).')
print('  (order_item_promotion toàn khóa — all-key ⇒ hiển nhiên đạt tới BCNF, không cần kiểm.)')


a) unit_price có phụ thuộc riêng product_id không?


  FD SAI       product_id -> unit_price        1509 vi phạm / 1,598 giá trị  


     trung vị 88 giá khác nhau/sản phẩm, tối đa 7,777
     → giá là giá TẠI THỜI ĐIỂM BÁN, cần CẢ khóa. Không phải phụ thuộc bộ phận.

b) promo_id có phụ thuộc riêng order_id không?
  FD ĐÚNG        order_id -> promo_id             0 vi phạm / 646,945 giá trị  (chỉ tính dòng khác NULL)


     NHƯNG 269 đơn có CẢ dòng mang promo lẫn dòng không mang ⇒ FD thất bại khi tính NULL.
     Trong 269 đơn đó, dòng MANG promo:
       promo có applicable_category      : 100%
       category sản phẩm == applicable   : 100%
     → promo giới hạn category chỉ dính vào dòng đủ điều kiện ⇒ promo là thuộc tính CẤP DÒNG.
       Junction table phải ở grain dòng hàng, không phải grain đơn.

c) discount_amount có phụ thuộc riêng order_id hay product_id không?


  FD SAI         order_id -> discount_amount  27771 vi phạm / 646,945 giá trị  
  FD SAI       product_id -> discount_amount   1455 vi phạm / 1,598 giá trị  
     → cần CẢ khóa (chiết khấu phụ thuộc promo × giá trị dòng). Không phải phụ thuộc bộ phận.

→ order_item KHÔNG vi phạm 2NF: cả 3 cột không khóa đều cần đủ (order_id, line_number).
  (order_item_promotion toàn khóa — all-key ⇒ hiển nhiên đạt tới BCNF, không cần kiểm.)


---
## 4 — 3NF: phụ thuộc bắc cầu (thuộc tính không khóa xác định thuộc tính không khóa)

### 4.1 `products`: `product_id → product_name → category, segment`

In [8]:
for c in ['category', 'segment']:
    show_fd(prod, 'product_name', c)
n_p, n_m = len(prod), prod.product_name.nunique()
print(f'\n  {n_p:,} sản phẩm nhưng chỉ {n_m:,} model ⇒ (category, segment) bị lặp thừa {n_p-n_m} lần')
print(f'  → tách product_model(product_name PK, category, segment)')

print(f'\n  Kiểm tra ngược: (product_name, size, color) có phải khóa không?')
print(f'    số dòng trùng: {prod.duplicated(["product_name","size","color"]).sum()} → KHÔNG')
print(f'    ⇒ product_id là surrogate CẦN THIẾT, không phải trang trí')
print(prod[prod.duplicated(['product_name','size','color'], keep=False)]
      .sort_values(['product_name','size','color']).head(4)
      [['product_id','product_name','size','color','price']].to_string(index=False))

  FD ĐÚNG    product_name -> category             0 vi phạm / 2,172 giá trị  
  FD ĐÚNG    product_name -> segment              0 vi phạm / 2,172 giá trị  

  2,412 sản phẩm nhưng chỉ 2,172 model ⇒ (category, segment) bị lặp thừa 240 lần
  → tách product_model(product_name PK, category, segment)

  Kiểm tra ngược: (product_name, size, color) có phải khóa không?
    số dòng trùng: 240 → KHÔNG
    ⇒ product_id là surrogate CẦN THIẾT, không phải trang trí
 product_id    product_name size color        price
        280 LotusWear UE-01    S   red 12596.850000
        380 LotusWear UE-01    S   red    34.036218
        281 LotusWear UE-02    M black 15746.850000
        381 LotusWear UE-02    M black    21.651582


### 4.2 `customers` và `orders`: địa lý lặp qua hai tầng

In [9]:
show_fd(cust, 'zip', 'city', '(trong customers)')
j = cust[['customer_id','zip','city']].merge(geo[['zip','city']], on='zip', suffixes=('_c','_g'))
print(f'  customers.city == geography.city qua zip : lệch {(j.city_c!=j.city_g).sum()} / {len(j):,}')
print('  → customer_id → zip → city là bắc cầu. Bỏ customers.city.\n')

j = orders[['order_id','customer_id','zip']].merge(cust[['customer_id','zip']], on='customer_id', suffixes=('_o','_c'))
print(f'  orders.zip == customers.zip : lệch {(j.zip_o!=j.zip_c).sum()} / {len(j):,}')
print('  → order_id → customer_id → zip là bắc cầu. Bỏ orders.zip.')
print('    (đồng thời xác nhận: KHÔNG có địa chỉ giao riêng trong dữ liệu)')

  FD ĐÚNG             zip -> city                 0 vi phạm / 31,491 giá trị  (trong customers)
  customers.city == geography.city qua zip : lệch 0 / 121,930
  → customer_id → zip → city là bắc cầu. Bỏ customers.city.



  orders.zip == customers.zip : lệch 0 / 646,945
  → order_id → customer_id → zip là bắc cầu. Bỏ orders.zip.
    (đồng thời xác nhận: KHÔNG có địa chỉ giao riêng trong dữ liệu)


### 4.3 `reviews`: FD đi theo chiều `review_title → rating`, KHÔNG phải ngược lại

Đây là điểm **sửa lại** so với `star_schema.md` §5.6, vốn phát biểu nhầm chiều.

In [10]:
ct = pd.crosstab(rev.rating, rev.review_title)
print('Số title mỗi rating dùng:')
print((ct > 0).sum(axis=1).to_string())
print(f'\nSố rating mà một title xuất hiện — tối đa: {(ct>0).sum(axis=0).max()}')
show_fd(rev, 'rating', 'review_title', '← chiều này SAI (mỗi rating có 3–4 title)')
show_fd(rev, 'review_title', 'rating', '← chiều này ĐÚNG')
print(f'\n  → review_id → review_title → rating là phụ thuộc bắc cầu.')
print(f'    Tách review_title_label(review_title PK, rating), {rev.review_title.nunique()} dòng.')

Số title mỗi rating dùng:
rating
1    4
2    3
3    3
4    4
5    4

Số rating mà một title xuất hiện — tối đa: 1
  FD SAI           rating -> review_title         5 vi phạm / 5 giá trị  ← chiều này SAI (mỗi rating có 3–4 title)
  FD ĐÚNG    review_title -> rating               0 vi phạm / 18 giá trị  ← chiều này ĐÚNG

  → review_id → review_title → rating là phụ thuộc bắc cầu.
    Tách review_title_label(review_title PK, rating), 18 dòng.


### 4.4 `payments` và `reviews`: cột sao chép từ bảng cha

In [11]:
j = orders[['order_id','payment_method']].merge(pay[['order_id','payment_method']], on='order_id', suffixes=('_o','_p'))
print(f'  payments.payment_method == orders.payment_method : lệch {(j.payment_method_o!=j.payment_method_p).sum()} / {len(j):,}')
j = rev[['order_id','customer_id']].merge(orders[['order_id','customer_id']], on='order_id', suffixes=('_r','_o'))
print(f'  reviews.customer_id  == orders.customer_id       : lệch {(j.customer_id_r!=j.customer_id_o).sum()} / {len(j):,}')
print('\n  → cả hai là order_id → (bảng cha) → thuộc tính. Bỏ khỏi bảng con.')

print(f'\n  promotions.promo_name unique: {promo.promo_name.is_unique} '
      f'({promo.promo_name.nunique()}/{len(promo)}) → chỉ là candidate key, KHÔNG có FD bắc cầu')

  payments.payment_method == orders.payment_method : lệch 0 / 646,945


  reviews.customer_id  == orders.customer_id       : lệch 0 / 113,551

  → cả hai là order_id → (bảng cha) → thuộc tính. Bỏ khỏi bảng con.

  promotions.promo_name unique: True (50/50) → chỉ là candidate key, KHÔNG có FD bắc cầu


### 4.5 `geography`: nơi 3NF sách vở và thiết kế tốt tách nhau

In [12]:
print(f'zip={geo.zip.nunique():,}  city={geo.city.nunique()}  district={geo.district.nunique()}  region={geo.region.nunique()}\n')
for a, b in [('zip','city'), ('zip','district'), ('city','region'), ('district','region'),
             ('city','district'), ('district','city')]:
    v, t = fd(geo, a, b)
    mx = geo.groupby(a)[b].nunique().max()
    ok = 'FD ĐÚNG' if v == 0 else 'FD SAI '
    print(f'  {ok}  {a:9s} → {b:9s}  {v:>3} vi phạm / {t:<6,}  (max {mx} giá trị)')
print('\n  → city và district ĐỀU roll-up về region nhưng CẮT CHÉO nhau (không lồng nhau).')
print('    3NF chặt: zip_area(zip, city, district) + city(city, region) + district(district, region)')
print('    ⇒ region tới được từ cùng một zip theo HAI đường, schema không ép hai đường khớp nhau.')

zip=39,948  city=42  district=39  region=3



  FD ĐÚNG  zip       → city         0 vi phạm / 39,948  (max 1 giá trị)
  FD ĐÚNG  zip       → district     0 vi phạm / 39,948  (max 1 giá trị)
  FD ĐÚNG  city      → region       0 vi phạm / 42      (max 1 giá trị)
  FD ĐÚNG  district  → region       0 vi phạm / 39      (max 1 giá trị)
  FD SAI   city      → district    42 vi phạm / 42      (max 19 giá trị)


  FD SAI   district  → city        39 vi phạm / 39      (max 16 giá trị)

  → city và district ĐỀU roll-up về region nhưng CẮT CHÉO nhau (không lồng nhau).
    3NF chặt: zip_area(zip, city, district) + city(city, region) + district(district, region)
    ⇒ region tới được từ cùng một zip theo HAI đường, schema không ép hai đường khớp nhau.


---
## 5 — Dư thừa dẫn xuất: nguyên tắc RIÊNG, không phải yêu cầu 3NF

3NF nói về **phụ thuộc hàm giữa các thuộc tính trong một quan hệ**.
Cột tính được từ cột khác (kể cả bảng khác) là dạng dư thừa *khác* — bỏ chúng vì
nguyên tắc "không lưu cái tính được", **không phải** vì chuẩn 3NF.

In [13]:
g = items.assign(_n=items.quantity*items.unit_price - items.discount_amount).groupby('order_id')._n.sum()
p = pay.set_index('order_id').payment_value
print(f'  payments.payment_value == Σ(qty×price − discount) : khớp {np.isclose(p, g.reindex(p.index)).mean()*100:.2f}%')
print(f'  → payments chỉ còn thông tin MỚI là installments\n')

m = (items.merge(orders[['order_id','order_date']], on='order_id')
          .merge(prod[['product_id','cogs']], on='product_id'))
a = m.assign(_r=m.quantity*m.unit_price, _c=m.quantity*m.cogs).groupby('order_date').agg(r=('_r','sum'), c=('_c','sum'))
j = sales.set_index('Date').join(a)
print(f'  sales.Revenue tái tạo từ order_items — sai số tương đối tối đa : {((j.Revenue-j.r).abs()/j.Revenue).max():.2e}')
print(f'  sales.COGS    tái tạo từ order_items — sai số tương đối tối đa : {((j.COGS-j.c).abs()/j.COGS).max():.2e}')
print('  → sales.csv là VIEW tổng hợp, không phải bảng cơ sở\n')

print('  inventory — cột nào thực sự tính được từ cột GIỮ LẠI?')
dim = inv.snapshot_date.dt.day
print(f'    stockout_flag == (stockout_days > 0)           : {(inv.stockout_flag == (inv.stockout_days>0).astype(int)).mean()*100:.1f}%  → BỎ')
print(f'    days_of_supply == stock_on_hand/(units_sold/30) : {np.isclose(inv.days_of_supply, inv.stock_on_hand/(inv.units_sold/30).replace([np.inf,-np.inf], np.nan), atol=1).mean()*100:.1f}%  → BỎ')
print(f'    fill_rate == 1 − stockout_days/số_ngày_tháng    : {np.isclose(inv.fill_rate, 1-inv.stockout_days/dim, atol=1e-4).mean()*100:.1f}%  → GIỮ')
st = inv.stock_on_hand + inv.units_received
print(f'    sell_through == units_sold/(soh+units_received) : {np.isclose(inv.sell_through_rate, inv.units_sold/st.replace(0,np.nan), atol=1e-3).mean()*100:.1f}%  → GIỮ')
print(f'    overstock_flag == (days_of_supply >= 90)        : {(inv.overstock_flag == (inv.days_of_supply>=90).astype(int)).mean()*100:.1f}%  → GIỮ')
print('\n  → chỉ 2/5 cột chứng minh được là dẫn xuất. 3 cột còn lại KHÔNG tính được từ cột giữ lại')
print('    ⇒ giữ chúng làm measure độc lập, đừng đoán công thức.')

  payments.payment_value == Σ(qty×price − discount) : khớp 100.00%
  → payments chỉ còn thông tin MỚI là installments



  sales.Revenue tái tạo từ order_items — sai số tương đối tối đa : 2.21e-16
  sales.COGS    tái tạo từ order_items — sai số tương đối tối đa : 1.03e-08
  → sales.csv là VIEW tổng hợp, không phải bảng cơ sở

  inventory — cột nào thực sự tính được từ cột GIỮ LẠI?
    stockout_flag == (stockout_days > 0)           : 100.0%  → BỎ
    days_of_supply == stock_on_hand/(units_sold/30) : 100.0%  → BỎ
    fill_rate == 1 − stockout_days/số_ngày_tháng    : 55.5%  → GIỮ
    sell_through == units_sold/(soh+units_received) : 65.9%  → GIỮ
    overstock_flag == (days_of_supply >= 90)        : 93.6%  → GIỮ

  → chỉ 2/5 cột chứng minh được là dẫn xuất. 3 cột còn lại KHÔNG tính được từ cột giữ lại
    ⇒ giữ chúng làm measure độc lập, đừng đoán công thức.


---
## 6 — Ràng buộc toàn vẹn cần khai báo (và vì sao "chỗ vi phạm" duy nhất là cảnh báo giả)

In [14]:
print(f'  đơn không có dòng hàng     : {(~orders.order_id.isin(items.order_id)).sum()}')
print(f'  đơn không có payment       : {(~orders.order_id.isin(pay.order_id)).sum()}')
print(f'  đơn không có shipment      : {(~orders.order_id.isin(ship.order_id)).sum():,}  → shipment là 1:0..1, FK phải NULLable')
print(f'  returns trùng (order_id, product_id) : {ret.duplicated(["order_id","product_id"]).sum()} dòng (return_id vẫn unique: {ret.return_id.is_unique})')
print(f'  reviews trùng (order_id, product_id) : {rev.duplicated(["order_id","product_id"]).sum()}')

dupk = set(map(tuple, items[items.duplicated(['order_id','product_id'], keep=False)][['order_id','product_id']].values))
print(f'\n  Hệ quả của 16 cặp khóa nhập nhằng — số dòng con không xác định được line_number:')
for n, d in [('returns', ret), ('reviews', rev)]:
    hit = sum(1 for t in map(tuple, d[['order_id','product_id']].values) if t in dupk)
    print(f'    {n:8s}: {hit} dòng → ETL phải gán line_number theo THỨ TỰ NGUỒN ổn định rồi mới join')

# --- return_quantity <= quantity: KẾT QUẢ PHỤ THUỘC VÀO KHÓA JOIN ---
# (a) join bằng (order_id, product_id) — KHÔNG phải khóa của order_item (§2.1)
z = ret.merge(items[['order_id','product_id','quantity']], on=['order_id','product_id'], how='left')
print(f'\n  (a) join (order_id, product_id) — khóa KHÔNG hợp lệ:')
print(f'      {len(z):,} dòng (fan-out +{len(z)-len(ret)}), vi phạm: {(z.return_quantity > z.quantity).sum()}')
print(z[z.return_quantity > z.quantity][['return_id','order_id','product_id','return_quantity','quantity']].to_string(index=False))

# (b) gán line_number theo THỨ TỰ NGUỒN (đúng quy tắc §2.1) rồi join bằng khóa thật
it = items.assign(line_number=items.groupby('order_id').cumcount() + 1)
it['_k'] = it.groupby(['order_id','product_id']).cumcount()
rt = ret.assign(_k=ret.groupby(['order_id','product_id']).cumcount())
z2 = rt.merge(it[['order_id','product_id','_k','line_number','quantity']],
              on=['order_id','product_id','_k'], how='left')
print(f'\n  (b) join (order_id, line_number) sau khi gán theo thứ tự nguồn:')
print(f'      {len(z2):,} dòng (không fan-out), vi phạm: {(z2.return_quantity > z2.quantity).sum()}')
print('      → RET-043492 là CẢNH BÁO GIẢ: return_quantity=3 bị ghép nhầm vào dòng quantity=1;')
print('        đơn đó có 2 dòng cùng sản phẩm, quantity = 4 và 1 ⇒ ghép đúng thì 3 ≤ 4 và 1 ≤ 1.')
print(f'\n  rating ngoài [1,5] : {(~rev.rating.between(1,5)).sum()}')

print('\n--- Toàn bộ CHECK trong DDL §8, kiểm trên dữ liệu nguồn ---')
checks = [
    ('product.unit_cogs <= list_price',        (prod.cogs > prod.price).sum(),                                    len(prod)),
    ('order_item.quantity > 0',                (items.quantity <= 0).sum(),                                       len(items)),
    ('order_item.discount_amount >= 0',        (items.discount_amount < 0).sum(),                                 len(items)),
    ('payment.installments > 0',               (pay.installments <= 0).sum(),                                     len(pay)),
    ('shipment.ship_date NOT NULL',            ship.ship_date.isna().sum(),                                       len(ship)),
    ('shipment.delivery >= ship_date',         (ship.delivery_date < ship.ship_date).sum(),                       len(ship)),
    ('product_return.return_quantity > 0',     (ret.return_quantity <= 0).sum(),                                  len(ret)),
    ('promotion.end_date >= start_date',       (promo.end_date < promo.start_date).sum(),                         len(promo)),
    ('review.rating BETWEEN 1 AND 5',          (~rev.rating.between(1, 5)).sum(),                                 len(rev)),
    ('web_traffic.unique_visitors <= sessions',(wt.unique_visitors > wt.sessions).sum(),                           len(wt)),
    ('review UNIQUE(order_id, product_id)',    rev.duplicated(['order_id', 'product_id']).sum(),                   len(rev)),
    ('return_qty <= qty — join (order_id,product_id)',  (z.return_quantity  > z.quantity).sum(),                   len(z)),
    ('return_qty <= qty — join (order_id,line_number)', (z2.return_quantity > z2.quantity).sum(),                  len(z2)),
]
for name, bad, tot in checks:
    mark = 'ĐẠT ' if bad == 0 else 'FAIL'
    print(f'  {mark}  {name:47s} {bad:>3} vi phạm / {tot:,}')
print('\n  → 13 dòng trên là 12 ràng buộc: hai dòng cuối là CÙNG một ràng buộc, khác khóa join.')
print('    12/12 ĐẠT khi join đúng khóa. Dòng FAIL KHÔNG phải lỗi dữ liệu mà là hệ quả của')
print('    khóa nhập nhằng (§2.1) — kiểm ràng buộc bằng khóa sai thì sinh ra lỗi không có thật.')


  đơn không có dòng hàng     : 0
  đơn không có payment       : 0
  đơn không có shipment      : 80,878  → shipment là 1:0..1, FK phải NULLable
  returns trùng (order_id, product_id) : 2 dòng (return_id vẫn unique: True)
  reviews trùng (order_id, product_id) : 0



  Hệ quả của 16 cặp khóa nhập nhằng — số dòng con không xác định được line_number:
    returns : 4 dòng → ETL phải gán line_number theo THỨ TỰ NGUỒN ổn định rồi mới join


    reviews : 2 dòng → ETL phải gán line_number theo THỨ TỰ NGUỒN ổn định rồi mới join



  (a) join (order_id, product_id) — khóa KHÔNG hợp lệ:
      39,943 dòng (fan-out +4), vi phạm: 1
 return_id  order_id  product_id  return_quantity  quantity
RET-043492    698128        1009                3         1



  (b) join (order_id, line_number) sau khi gán theo thứ tự nguồn:
      39,939 dòng (không fan-out), vi phạm: 0
      → RET-043492 là CẢNH BÁO GIẢ: return_quantity=3 bị ghép nhầm vào dòng quantity=1;
        đơn đó có 2 dòng cùng sản phẩm, quantity = 4 và 1 ⇒ ghép đúng thì 3 ≤ 4 và 1 ≤ 1.

  rating ngoài [1,5] : 0

--- Toàn bộ CHECK trong DDL §8, kiểm trên dữ liệu nguồn ---
  ĐẠT   product.unit_cogs <= list_price                   0 vi phạm / 2,412
  ĐẠT   order_item.quantity > 0                           0 vi phạm / 714,669
  ĐẠT   order_item.discount_amount >= 0                   0 vi phạm / 714,669
  ĐẠT   payment.installments > 0                          0 vi phạm / 646,945
  ĐẠT   shipment.ship_date NOT NULL                       0 vi phạm / 566,067
  ĐẠT   shipment.delivery >= ship_date                    0 vi phạm / 566,067
  ĐẠT   product_return.return_quantity > 0                0 vi phạm / 39,939
  ĐẠT   promotion.end_date >= start_date                  0 vi phạm / 50
  ĐẠT 

---
## 7 — Quy mô mô hình chuẩn hóa

In [15]:
rows = [
    ('region',                 geo.region.nunique(),   'tách từ geography'),
    ('city',                   geo.city.nunique(),     'tách từ geography'),
    ('district',               geo.district.nunique(), 'tách từ geography'),
    ('zip_area',               len(geo),               'geography còn lại'),
    ('customer',               len(cust),              'bỏ city'),
    ('product_model',          prod.product_name.nunique(), 'MỚI — tách 3NF'),
    ('product',                len(prod),              'bỏ category, segment'),
    ('promotion',              len(promo),             '1:1'),
    ('"order"',                len(orders),            'bỏ zip'),
    ('order_item',             len(items),             '+ line_number, bỏ 2 cột promo'),
    ('order_item_promotion',   int(items.promo_id.notna().sum() + items.promo_id_2.notna().sum()), 'MỚI — tách 1NF'),
    ('payment',                len(pay),               'chỉ còn installments'),
    ('shipment',               len(ship),              '1:0..1'),
    ('product_return',         len(ret),               '1:1'),
    ('review_title_label',     rev.review_title.nunique(), 'MỚI — tách 3NF'),
    ('review',                 len(rev),               'bỏ customer_id, rating'),
    ('inventory_snapshot',     len(inv),               '17 → 9 cột (3 cột 2NF + 2 dẫn xuất từ ngày + 2 dẫn xuất + 1 chết)'),
    ('web_traffic',            len(wt),                '1:1'),
    ('daily_sales_forecast',   len(sub),               'sample_submission'),
]
print(f'{"bảng":24s}{"số dòng":>10s}   ghi chú')
print('-'*70)
for n, k, note in rows:
    print(f'{n:24s}{k:>10,}   {note}')
print('-'*70)
print(f'{"TỔNG":24s}{sum(k for _,k,_ in rows):>10,}   19 bảng (sales.csv = view, không lưu)')

bảng                       số dòng   ghi chú
----------------------------------------------------------------------
region                           3   tách từ geography
city                            42   tách từ geography
district                        39   tách từ geography
zip_area                    39,948   geography còn lại
customer                   121,930   bỏ city
product_model                2,172   MỚI — tách 3NF
product                      2,412   bỏ category, segment
promotion                       50   1:1
"order"                    646,945   bỏ zip
order_item                 714,669   + line_number, bỏ 2 cột promo
order_item_promotion       276,522   MỚI — tách 1NF
payment                    646,945   chỉ còn installments
shipment                   566,067   1:0..1
product_return              39,939   1:1
review_title_label              18   MỚI — tách 3NF
review                     113,551   bỏ customer_id, rating
inventory_snapshot          60,247   17 → 9 cột (3

---
## 8 — Kết luận

| Chuẩn | Vi phạm | Nguyên nhân |
|---|---|---|
| **1NF** | 2 vi phạm, cùng ở `order_items` | không định danh được dòng; nhóm lặp `promo_id`/`promo_id_2` |
| **2NF** | 1 bảng | `inventory` — bảng **duy nhất** có khóa tổ hợp; vi phạm ở cả hai vế khóa |
| **3NF** | 5 bảng + 1 judgment call | `products`, `customers`, `orders`, `reviews`, `payments`; và `geography` (`zip → city → region`) |

Ba điểm cần nhớ:

1. **2NF hiếm gặp vì lý do cấu trúc** — nó chỉ tồn tại khi có khóa tổ hợp. 1/14 bảng ở đây.
2. **`review_title → rating`, không phải chiều ngược lại** — sửa lại `star_schema.md` §5.6.
3. **Bỏ cột dẫn xuất ≠ chuẩn hóa.** Hai nguyên tắc khác nhau, hay bị gộp làm một.

Xem `normalized_schema.md` cho ERD, DDL và phần "khi nào dùng chuẩn nào".